# A2 — Cloud Misconfiguration Rule Catalog (L2)

Shallow-clones Prowler, Checkov and ScoutSuite, extracts every check definition, normalises them into one schema, and writes `data/processed/l2_misconfig_catalog.csv`. All logic lives in `src/misconfig/`; this notebook is a thin runner (CLAUDE.md rule 9).

**Severity caveat (matters for A3):** Prowler ships a full five-level severity, ScoutSuite only `danger`/`warning`, and Checkov OSS ships none at all (severities live in the commercial Bridgecrew platform). See the mapping documented in `src/misconfig/normaliser.py`.

In [1]:
import os
import sys
from pathlib import Path

# Every src/ path is relative to the project root (data/..., outputs/...).
# Without this chdir a run from notebooks/ writes into notebooks/data/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.misconfig.normaliser import build_l2_catalog, log_stats

In [2]:
df = build_l2_catalog()
log_stats(df)

A2 pipeline:   0%|          | 0/5 [00:00<?, ?stage/s]

Cloning/locating repos:   0%|          | 0/5 [00:00<?, ?stage/s]

2026-11-23 17:49:00,902 INFO src.misconfig.normaliser: Repo commits: {'prowler': '806be2d', 'checkov': 'd8aec9d', 'scoutsuite': '7909f2f'}


Extracting Prowler:  20%|██        | 1/5 [00:00<00:00, 20.44stage/s]

Prowler checks:   0%|          | 0/1608 [00:00<?, ?check/s]

Prowler checks:  26%|██▋       | 426/1608 [00:00<00:00, 4225.39check/s]

... (22 outputs left out)

Normalising and writing CSV: 100%|██████████| 5/5 [00:01<00:00,  3.19stage/s]

2026/11/23 17:49:03 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/spopa/Documents/Internship Project/internship-project/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


2026-11-23 17:49:03,405 INFO src.tracking: Logged A2 to MLflow (run bf535bdc994d468f9acd841321e3ba19)



A2 extraction complete: data/processed/l2_misconfig_catalog.csv
Total checks:            3,870

Per tool:
  checkov       1,684   (0% with a usable severity)
  prowler       1,608   (100% with a usable severity)
  scoutsuite      578   (92% with a usable severity)

Severity distribution (normalised):
  unknown          1,731 (44.7%)
  medium           1,128 (29.1%)
  high               752 (19.4%)
  low                137 (3.5%)
  critical           120 (3.1%)
  informational        2 (0.1%)

Top 10 providers:
  aws              1,383
  azure              575
  gcp                391
  kubernetes         307
  (none)             288
  m365               144
  k8s                113
  googleworkspace     65
  alibabacloud        65
  oraclecloud         52

Distinct services:       474
With remediation text:   46.3%

Note: Checkov OSS ships no severities (they live in the commercial
platform), and ScoutSuite has only danger/warning. See the severity
mapping in this module's docstring b

In [3]:
df.head()

,catalog_id,check_id,tool,provider,service,title,description,rationale_risk,severity_raw,severity_normalised,categories,resource_type,framework,check_type,remediation_text,remediation_code,remediation_url,references,source_file,source_commit
0,checkov:CKV2_ANSIBLE_1,CKV2_ANSIBLE_1,checkov,,tasks.ansible.builtin.uri,Ensure that HTTPS url is used with uri,None,None,None,unknown,NETWORKING,tasks.ansible.builtin.uri|tasks.uri,ansible,graph_checks,None,None,None,,checkov/ansible/checks/graph_checks/UriHttpsOn...,d8aec9d
1,checkov:CKV2_ANSIBLE_2,CKV2_ANSIBLE_2,checkov,,,Ensure that HTTPS url is used with get_url,None,None,None,unknown,NETWORKING,,ansible,graph_checks,None,None,None,,checkov/ansible/checks/graph_checks/GetUrlHttp...,d8aec9d
2,checkov:CKV2_ANSIBLE_3,CKV2_ANSIBLE_3,checkov,,block,Ensure block is handling task errors properly,None,None,None,unknown,BACKUP_AND_RECOVERY,block,ansible,graph_checks,None,None,None,,checkov/ansible/checks/graph_checks/BlockError...,d8aec9d
3,checkov:CKV2_ANSIBLE_4,CKV2_ANSIBLE_4,checkov,,tasks.ansible.builtin.dnf,Ensure that packages with untrusted or missing...,None,None,None,unknown,GENERAL_SECURITY,tasks.ansible.builtin.dnf|tasks.dnf,ansible,graph_checks,None,None,None,,checkov/ansible/checks/graph_checks/DnfDisable...,d8aec9d
4,checkov:CKV2_ANSIBLE_5,CKV2_ANSIBLE_5,checkov,,tasks.ansible.builtin.dnf,Ensure that SSL validation isn't disabled with...,None,None,None,unknown,GENERAL_SECURITY,tasks.ansible.builtin.dnf|tasks.dnf,ansible,graph_checks,None,None,None,,checkov/ansible/checks/graph_checks/DnfSslVeri...,d8aec9d


## Cross-tool coverage — what A3 has to work with

A3 compares severities across tools, so it needs checks that more than one tool covers. Checkov contributes no severity, which constrains that analysis to Prowler vs ScoutSuite unless a proxy is used.

In [4]:
coverage = (
    df[df.provider == "aws"]
    .groupby("service")
    .agg(tools=("tool", "nunique"), checks=("catalog_id", "count"))
    .sort_values(["tools", "checks"], ascending=False)
)
coverage.head(15)

,tools,checks
service,,
iam,3,108
ec2,3,105
rds,3,59
s3,3,44
cloudwatch,3,42
cloudtrail,3,30
redshift,3,28
cloudfront,3,27
vpc,3,19
